# KYC/AML Industry Demo – Full Walkthrough

An end-to-end **Know Your Customer / Anti-Money Laundering** demo built on Databricks. It covers AI-powered customer onboarding, real-time transaction screening with ML, and a Chief Compliance Officer dashboard — all wired together with Unity Catalog, Databricks Apps, and Lakeflow.

**Audience:** Solutions Architects, Data Engineers, Compliance / Risk teams  
**Duration:** ~30 min (live walkthrough) or self-paced

---

## Architecture

| Layer | Component | Key Tech |
|-------|-----------|----------|
| **Frontend** | 6-scene interactive app | Databricks Apps, HTML/JS |
| **Agents** | KYC onboarding agent (Scene 1) | LangChain, UC Functions, Lakebase, Databricks Apps |
| **ML** | SEPA fraud/risk classifier (Scene 2) | scikit-learn, MLflow, Unity Catalog Models |
| **Streaming** | Real-time transaction pipeline (Scene 2) | Zerobus Ingest, Delta Live Tables |
| **Data** | Unity Catalog tables, volumes, functions | Delta Lake, UC |
| **Dashboards** | CCO compliance dashboard (Scene 6) | AI/BI Dashboards, Genie Spaces |

## Prerequisites

1. **Configure [`_shared/config`](./_shared/config)** — open this file and set your **catalog**, **schema**, table name, and pipeline name. All notebooks read from it via `%run`.
2. **Databricks workspace** with Unity Catalog enabled
3. **Deploy the bundle:** `databricks bundle deploy --target dev`
4. **Catalog permissions:** Ability to create schemas and tables in the configured catalog
5. **(Scene 2)** Zerobus Ingest enabled + service principal with MODIFY/SELECT on the target table
6. **(Scene 1 deploy)** SQL warehouse ID for the Statement Execution API

---

## Scene 1: KYC Onboarding (AI Agent)

An AI agent walks a customer through onboarding: identity verification (passport parsing via `ai_parse_document`), proof of address, PEP/sanctions/fraud screening, SBI risk check, and account creation — all backed by Unity Catalog functions.

| # | Notebook | Purpose |
|---|----------|---------|
| 1 | [`01_kyc_data_setup`](./scene1_kyc_onboarding/01_kyc_data_setup) | Create schema, seed tables (PEP, sanctions, fraud, SBI codes), and UC functions |
| 2 | [`02_kyc_agent`](./scene1_kyc_onboarding/02_kyc_agent) | Run the KYC agent interactively in the notebook (Spark session) |
| 3 | [`03_kyc_agent_deploy`](./scene1_kyc_onboarding/03_kyc_agent_deploy) | Deploy the agent as a Databricks App with Lakebase memory |

**Run order:** `01` (once) -> `02` (interactive demo) or `03` (deploy as app)

---

## Scene 2: Transaction Screening (ML + Streaming)

Train a fraud/risk ML model on SEPA transactions, deploy a DLT streaming pipeline, and ingest real-time data via Zerobus. Transactions flow through bronze -> silver (ML risk scoring) -> gold layers.

| # | Notebook | Purpose |
|---|----------|---------|
| 1 | [`01_train_model_and_pipeline`](./scene2_txn_screening/01_train_model_and_pipeline) | Generate training data, train classifier, register model in UC, create SEPA source table, link DLT pipeline |
| 2 | [`02_zerobus_ingest`](./scene2_txn_screening/02_zerobus_ingest) | Stream SEPA transactions into the source table via Zerobus Ingest |

**Run order:** `01` (train + pipeline) -> `02` (ingest)

---

## Scenes 3–5: AI Monitoring, EDD Investigation, SAR Filing

These scenes are **frontend-only** (no notebooks required). They are part of the interactive app deployed via `databricks.yml` and demonstrate:

- **Scene 3** — AI-powered transaction monitoring alerts
- **Scene 4** — Enhanced Due Diligence investigation workflow
- **Scene 5** — Suspicious Activity Report (SAR) filing

---

## Scene 6: CCO Dashboard

Generate compliance metrics data for the Chief Compliance Officer dashboard.

| # | Notebook | Purpose |
|---|----------|---------|
| 1 | [`01_generate_dashboard_data`](./scene6_cco_dashboard/01_generate_dashboard_data) | Generate `compliance_metrics` (20k rows) and `kpi_summary` tables |

After generating data, create an AI/BI Dashboard or Genie Space on top of these tables and embed it in the frontend app.

---

## Shared Utilities

| File | Purpose |
|------|---------|
| [`_shared/config`](./_shared/config) | Catalog, schema, and derived variables for all notebooks. When run as a **job**, `databricks.yml` variables override automatically. |
| [`_shared/sepa_constants`](./_shared/sepa_constants) | SEPA transaction constants (companies, locations, currencies) used by ML training and Zerobus ingest |

---

## Deployment & Cleanup

**Deploy everything:**
```bash
databricks bundle deploy --target dev
```

**Cleanup:** Each notebook with setup cells has a corresponding "Delete resources" section at the bottom. Run those cells to drop tables, functions, volumes, apps, and pipelines created during the demo.

For a full teardown, also run:
```bash
databricks bundle destroy --target dev
```